# 82. The Single Facility Location Problem
## Tier 2: The Classic Heuristic (Geometric Iterative Refinement)

### Key Assumptions
- Transportation cost is proportional to Euclidean distance
- Facility location must satisfy feasibility constraints
- Discrete candidate locations are considered around the center of gravity
- Local improvement can enhance the initial solution

### Approach (Step-by-Step)
The geometric iterative refinement heuristic improves upon the basic center of gravity:
1. **Compute initial center of gravity** as starting point
2. **Generate candidate locations** on a grid around the center
3. **Apply feasibility filters** to eliminate impractical locations
4. **Evaluate candidates** using weighted distance calculation
5. **Select best candidate** as the initial solution
6. **Apply local improvement** using hill climbing in 8 directions
7. **Iterate until no improvement** is possible

### What to Look For in the Results
- Comparison with mathematical optimum (Tier 1)
- Impact of feasibility constraints on solution quality
- Convergence behavior of the iterative refinement
- Trade-off between optimality and practical feasibility

### Concrete Example (MegaCorp with Constraints)
**Enhanced Problem Context:**
MegaCorp's distribution center location must consider:
- **Restricted area**: Zone (x≥8, y≥4) is unavailable (environmental constraints)
- **Boundary limits**: Facility must be within coordinates (1≤x≤12, 1≤y≤10)
- **Discretization**: Location must be practical for construction
- **Local improvement**: Fine-tuning within feasible region

In [1]:
# Import required libraries for heuristic implementation and visualization
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set plotting style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class DemandLocation:
    """Class to represent a demand location with coordinates and weight"""
    id: int
    x: float
    y: float
    demand: float
    
    def distance_to(self, x: float, y: float) -> float:
        """Calculate Euclidean distance to a point"""
        return np.sqrt((self.x - x)**2 + (self.y - y)**2)

# Define MegaCorp's store locations and demands (same as Tier 1)
stores = [
    DemandLocation(id=1, x=2, y=3, demand=150),
    DemandLocation(id=2, x=8, y=1, demand=200),
    DemandLocation(id=3, x=6, y=7, demand=180),
    DemandLocation(id=4, x=12, y=4, demand=160),
    DemandLocation(id=5, x=4, y=9, demand=110)
]

print("MegaCorp Store Locations (Same as Tier 1):")
for store in stores:
    print(f"Store {store.id}: ({store.x}, {store.y}) with demand {store.demand} units")

total_demand = sum(store.demand for store in stores)
print(f"\nTotal Weekly Demand: {total_demand} units")

In [ ]:
def is_feasible(x: float, y: float) -> bool:
    """
    Check if a location is feasible given practical constraints.
    
    Constraints:
    1. Restricted area: x≥8 and y≥4 is unavailable (environmental zone)
    2. Boundary limits: 1≤x≤12 and 1≤y≤10 (operational area)
    3. Additional constraints can be added here
    """
    # Constraint 1: Avoid restricted environmental zone
    if x >= 8 and y >= 4:
        return False
    
    # Constraint 2: Boundary check for operational limits
    if not (1 <= x <= 12 and 1 <= y <= 10):
        return False
    
    # Future constraints could be added here:
    # - Distance from highways
    # - Zoning restrictions
    # - Terrain considerations
    
    return True

def calculate_center_of_gravity(locations: List[DemandLocation]) -> Tuple[float, float]:
    """
    Calculate the center of gravity (same as Tier 1) for initial estimate.
    """
    total_demand = sum(store.demand for store in locations)
    weighted_x = sum(store.x * store.demand for store in locations)
    weighted_y = sum(store.y * store.demand for store in locations)
    
    optimal_x = weighted_x / total_demand
    optimal_y = weighted_y / total_demand
    
    return optimal_x, optimal_y

# Test feasibility function with key points
print("Feasibility Testing:")
print("=" * 40)
test_points = [(6.675, 4.425), (8.5, 5), (7, 3), (0.5, 5), (12.5, 8)]
for x, y in test_points:
    feasible = is_feasible(x, y)
    print(f"Point ({x}, {y}): {'✅ Feasible' if feasible else '❌ Not feasible'}")

In [ ]:
def generate_grid_candidates(center_x: float, center_y: float, 
                            grid_resolution: float = 0.5,
                            search_radius: int = 2) -> List[Tuple[float, float]]:
    """
    Generate candidate locations on a grid around the center of gravity.
    
    Parameters:
    - center_x, center_y: Center of gravity coordinates
    - grid_resolution: Step size for grid (default 0.5)
    - search_radius: Radius around center to search (default 2)
    
    Returns:
    - List of feasible (x, y) candidate locations
    """
    candidates = []
    
    # Generate grid points around center
    x_start = center_x - search_radius
    x_end = center_x + search_radius
    y_start = center_y - search_radius
    y_end = center_y + search_radius
    
    # Create grid with specified resolution
    x_points = np.arange(x_start, x_end + grid_resolution, grid_resolution)
    y_points = np.arange(y_start, y_end + grid_resolution, grid_resolution)
    
    print(f"\nGenerating Grid Candidates:")
    print(f"Center: ({center_x:.3f}, {center_y:.3f})")
    print(f"Grid resolution: {grid_resolution}")
    print(f"Search radius: {search_radius}")
    print(f"X range: [{x_start:.1f}, {x_end:.1f}]")
    print(f"Y range: [{y_start:.1f}, {y_end:.1f}]")
    
    # Generate all grid points and filter by feasibility
    total_points = 0
    feasible_points = 0
    
    for x in x_points:
        for y in y_points:
            total_points += 1
            if is_feasible(x, y):
                candidates.append((x, y))
                feasible_points += 1
    
    print(f"Total grid points generated: {total_points}")
    print(f"Feasible candidates: {feasible_points}")
    print(f"Feasibility rate: {(feasible_points/total_points)*100:.1f}%")
    
    return candidates

# Calculate center of gravity and generate candidates
center_x, center_y = calculate_center_of_gravity(stores)
candidates = generate_grid_candidates(center_x, center_y)

print(f"\nCenter of Gravity: ({center_x:.3f}, {center_y:.3f})")
print(f"Is center feasible? {'Yes' if is_feasible(center_x, center_y) else 'No'}")
print(f"Generated {len(candidates)} feasible candidates")

In [ ]:
def calculate_weighted_distance(locations: List[DemandLocation], 
                                facility_x: float, facility_y: float) -> float:
    """
    Calculate total weighted distance from facility to all demand locations.
    """
    total_cost = 0.0
    for store in locations:
        distance = store.distance_to(facility_x, facility_y)
        weighted_cost = store.demand * distance
        total_cost += weighted_cost
    return total_cost

def evaluate_candidates(locations: List[DemandLocation], 
                        candidates: List[Tuple[float, float]]) -> Tuple[Tuple[float, float], float]:
    """
    Evaluate all candidate locations and select the best one.
    
    Returns:
    - Best candidate location (x, y)
    - Best candidate cost
    """
    print("\nEvaluating Candidates:")
    print("=" * 50)
    
    best_candidate = None
    best_cost = float('inf')
    
    # Evaluate each candidate
    for i, (x, y) in enumerate(candidates):
        cost = calculate_weighted_distance(locations, x, y)
        
        if cost < best_cost:
            best_cost = cost
            best_candidate = (x, y)
        
        # Show progress for first few and best candidates
        if i < 5 or (x, y) == best_candidate:
            status = "⭐ BEST" if (x, y) == best_candidate else ""
            print(f"Candidate ({x:.1f}, {y:.1f}): Cost = {cost:.2f} {status}")
    
    print(f"\nBest grid candidate: ({best_candidate[0]:.3f}, {best_candidate[1]:.3f}) with cost {best_cost:.2f}")
    
    return best_candidate, best_cost

# Evaluate candidates and find best grid location
best_grid_candidate, grid_cost = evaluate_candidates(stores, candidates)

In [ ]:
def hill_climbing_improvement(locations: List[DemandLocation], 
                             initial_location: Tuple[float, float],
                             step_size: float = 0.1,
                             max_iterations: int = 100) -> Tuple[Tuple[float, float], float, List]:
    """
    Apply hill climbing local improvement to refine the solution.
    
    Parameters:
    - locations: List of demand locations
    - initial_location: Starting point for improvement
    - step_size: Step size for local search
    - max_iterations: Maximum iterations to prevent infinite loops
    
    Returns:
    - Final improved location (x, y)
    - Final cost
    - History of improvements
    """
    print("\nLocal Improvement via Hill Climbing:")
    print("=" * 40)
    
    # Define 8 directions for movement (N, NE, E, SE, S, SW, W, NW)
    directions = [
        (0, step_size),      # North
        (step_size, step_size),   # Northeast
        (step_size, 0),      # East
        (step_size, -step_size),  # Southeast
        (0, -step_size),     # South
        (-step_size, -step_size), # Southwest
        (-step_size, 0),     # West
        (-step_size, step_size)   # Northwest
    ]
    
    current_location = list(initial_location)
    current_cost = calculate_weighted_distance(locations, current_location[0], current_location[1])
    improvement_history = [(current_location[0], current_location[1], current_cost)]
    
    print(f"Starting location: ({current_location[0]:.3f}, {current_location[1]:.3f})")
    print(f"Initial cost: {current_cost:.2f}")
    
    iteration = 0
    improvements_made = 0
    
    while iteration < max_iterations:
        iteration += 1
        improved = False
        
        # Try all 8 directions
        for dx, dy in directions:
            new_location = [current_location[0] + dx, current_location[1] + dy]
            
            # Check feasibility
            if not is_feasible(new_location[0], new_location[1]):
                continue
            
            # Calculate cost for new location
            new_cost = calculate_weighted_distance(locations, new_location[0], new_location[1])
            
            # If improvement found, move to new location
            if new_cost < current_cost:
                current_location = new_location
                current_cost = new_cost
                improvement_history.append((current_location[0], current_location[1], current_cost))
                improved = True
                improvements_made += 1
                
                print(f"Iteration {iteration}: Improved to ({current_location[0]:.3f}, {current_location[1]:.3f}), Cost: {current_cost:.2f}")
                break  # Move to next iteration with new position
        
        # If no improvement in any direction, stop
        if not improved:
            print(f"No improvement found in iteration {iteration}. Stopping.")
            break
    
    print(f"\nHill Climbing Results:")
    print(f"Total iterations: {iteration}")
    print(f"Improvements made: {improvements_made}")
    print(f"Final location: ({current_location[0]:.3f}, {current_location[1]:.3f})")
    print(f"Final cost: {current_cost:.2f}")
    print(f"Total improvement: {((grid_cost - current_cost) / grid_cost) * 100:.2f}%")
    
    return tuple(current_location), current_cost, improvement_history

# Apply hill climbing improvement
final_location, final_cost, improvement_history = hill_climbing_improvement(stores, best_grid_candidate)

In [ ]:
def compare_with_mathematical_optimum():
    """
    Compare heuristic solution with mathematical optimum from Tier 1.
    """
    print("\n" + "="*60)
    print("COMPARISON WITH MATHEMATICAL OPTIMUM (Tier 1)")
    print("="*60)
    
    # Mathematical optimum (from Tier 1)
    math_opt_x, math_opt_y = 6.675, 4.425
    math_opt_cost = 3367  # From Tier 1 calculations
    
    # Check if mathematical optimum is feasible
    math_feasible = is_feasible(math_opt_x, math_opt_y)
    
    print(f"Mathematical Optimum: ({math_opt_x:.3f}, {math_opt_y:.3f})")
    print(f"Mathematical Optimum Cost: {math_opt_cost:.2f}")
    print(f"Mathematical Optimum Feasible: {'Yes' if math_feasible else 'No'}")
    
    print(f"\nHeuristic Solution: ({final_location[0]:.3f}, {final_location[1]:.3f})")
    print(f"Heuristic Solution Cost: {final_cost:.2f}")
    print(f"Heuristic Solution Feasible: Yes (by construction)")
    
    # Calculate performance metrics
    if not math_feasible:
        print("\n🎯 KEY INSIGHT:")
        print("Mathematical optimum is infeasible due to constraints!")
        print("Heuristic provides the best FEASIBLE solution.")
        
        # Calculate cost if we forced the mathematical optimum
        forced_math_cost = calculate_weighted_distance(stores, math_opt_x, math_opt_y)
        print(f"\nIf mathematical optimum were allowed: {forced_math_cost:.2f}")
        print(f"Heuristic premium for feasibility: {((final_cost - forced_math_cost) / forced_math_cost) * 100:.2f}%")
    else:
        cost_gap = ((final_cost - math_opt_cost) / math_opt_cost) * 100
        print(f"\nOptimality Gap: {cost_gap:.2f}%")
        
        if cost_gap < 1:
            print("✅ Excellent: Heuristic within 1% of optimum!")
        elif cost_gap < 5:
            print("✅ Good: Heuristic within 5% of optimum!")
        else:
            print("⚠️  Moderate gap: Consider refinement parameters")

# Compare with mathematical optimum
compare_with_mathematical_optimum()

In [ ]:
def visualize_heuristic_solution():
    """
    Create comprehensive visualization of the heuristic solution process.
    """
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('P82-Tier-2: Geometric Iterative Refinement Heuristic', fontsize=16, fontweight='bold')
    
    # Plot 1: Solution Map with Constraints
    ax1.set_title('Heuristic Solution with Constraints', fontweight='bold')
    
    # Plot demand locations
    for store in stores:
        circle_size = store.demand / 8
        ax1.scatter(store.x, store.y, s=circle_size, alpha=0.7, 
                   label=f'Store {store.id}', edgecolors='black', linewidth=2)
        ax1.annotate(f'S{store.id}', (store.x, store.y), xytext=(3, 3), 
                    textcoords='offset points', fontsize=9, fontweight='bold')
    
    # Plot mathematical optimum (from Tier 1)
    math_opt_x, math_opt_y = 6.675, 4.425
    ax1.scatter(math_opt_x, math_opt_y, s=200, c='gold', marker='o', 
               label='Math Optimum', edgecolors='black', linewidth=2)
    ax1.annotate('Math\nOptimum', (math_opt_x, math_opt_y), xytext=(5, 5), 
                textcoords='offset points', fontsize=8, 
                bbox=dict(boxstyle='round,pad=0.3', facecolor='gold', alpha=0.7))
    
    # Plot heuristic solution
    ax1.scatter(final_location[0], final_location[1], s=300, c='red', marker='*', 
               label='Heuristic Solution', edgecolors='black', linewidth=2)
    ax1.annotate('Heuristic\nSolution', final_location, xytext=(5, 5), 
                textcoords='offset points', fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='red', alpha=0.3))
    
    # Show restricted area
    x_restricted = np.linspace(8, 12, 50)
    y_restricted = np.linspace(4, 10, 50)
    X_restricted, Y_restricted = np.meshgrid(x_restricted, y_restricted)
    ax1.contourf(X_restricted, Y_restricted, levels=1, colors=['red'], alpha=0.2)
    ax1.text(10, 7, 'RESTRICTED\nAREA', ha='center', va='center', 
             fontsize=12, fontweight='bold', color='red', alpha=0.7)
    
    # Plot grid candidates
    if candidates:
        cand_x, cand_y = zip(*candidates)
        ax1.scatter(cand_x, cand_y, s=20, c='lightblue', alpha=0.5, 
                   label='Grid Candidates', marker='s')
    
    ax1.set_xlabel('X Coordinate')
    ax1.set_ylabel('Y Coordinate')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper left', bbox_to_anchor=(1, 1))
    
    # Plot 2: Improvement History
    ax2.set_title('Hill Climbing Improvement History', fontweight='bold')
    if improvement_history:
        iterations = range(len(improvement_history))
        costs = [record[2] for record in improvement_history]
        
        ax2.plot(iterations, costs, 'b-o', linewidth=2, markersize=6)
        ax2.set_xlabel('Iteration')
        ax2.set_ylabel('Total Weighted Distance')
        ax2.grid(True, alpha=0.3)
        
        # Annotate start and end points
        ax2.annotate(f'Start: {costs[0]:.1f}', (0, costs[0]), 
                    xytext=(10, 10), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen'))
        ax2.annotate(f'Final: {costs[-1]:.1f}', (len(costs)-1, costs[-1]), 
                    xytext=(-10, -20), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightcoral'))
    
    # Plot 3: Cost Comparison
    ax3.set_title('Solution Cost Comparison', fontweight='bold')
    methods = ['Mathematical\nOptimum', 'Best Grid\nCandidate', 'Final\nHeuristic']
    costs = [3367, grid_cost, final_cost]  # Mathematical optimum from Tier 1
    colors = ['gold', 'lightblue', 'red']
    
    bars = ax3.bar(methods, costs, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
    ax3.set_ylabel('Total Weighted Distance')
    ax3.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, cost in zip(bars, costs):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + 20,
                f'{cost:.0f}', ha='center', va='bottom', fontweight='bold')
    
    # Plot 4: Algorithm Performance
    ax4.set_title('Algorithm Performance Metrics', fontweight='bold')
    
    # Create performance metrics
    metrics_text = f"""
    ALGORITHM PERFORMANCE SUMMARY
    =============================
    
    🎯 SOLUTION QUALITY:
       Final Location: ({final_location[0]:.3f}, {final_location[1]:.3f})
       Final Cost: {final_cost:.2f}
       Feasibility: ✅ GUARANTEED
    
    🔍 SEARCH PROCESS:
       Grid Candidates Generated: {len(candidates)}
       Best Grid Cost: {grid_cost:.2f}
       Hill Climbing Improvements: {len(improvement_history)-1}
       Total Improvement: {((grid_cost - final_cost) / grid_cost) * 100:.2f}%
    
    ⚡ COMPUTATIONAL EFFICIENCY:
       Grid Evaluation: O(n²) where n = candidates
       Local Search: O(k) where k = iterations
       Total Time: < 1 second for typical problems
    
    🏆 PRACTICAL ADVANTAGES:
       ✅ Handles real-world constraints
       ✅ Produces implementable solutions
       ✅ Transparent decision process
       ✅ Customizable feasibility rules
    """
    
    ax4.axis('off')
    ax4.text(0.1, 0.9, metrics_text, transform=ax4.transAxes, fontsize=10,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

# Create comprehensive visualization
visualize_heuristic_solution()

In [ ]:
def test_robustness_scenarios():
    """
    Test the heuristic's robustness under different scenarios.
    """
    print("\n" + "="*60)
    print("ROBUSTNESS TESTING UNDER DIFFERENT SCENARIOS")
    print("="*60)
    
    scenarios = [
        {
            'name': 'Base Case',
            'locations': stores,
            'description': 'Original MegaCorp configuration'
        },
        {
            'name': 'High Demand Store 1',
            'locations': [
                DemandLocation(id=1, x=2, y=3, demand=300),  # Increased
                DemandLocation(id=2, x=8, y=1, demand=200),
                DemandLocation(id=3, x=6, y=7, demand=180),
                DemandLocation(id=4, x=12, y=4, demand=160),
                DemandLocation(id=5, x=4, y=9, demand=110)
            ],
            'description': 'Store 1 demand doubled'
        },
        {
            'name': 'Additional Store',
            'locations': stores + [DemandLocation(id=6, x=10, y=8, demand=200)],
            'description': 'New high-demand store added'
        },
        {
            'name': 'Tighter Constraints',
            'locations': stores,
            'description': 'Smaller feasible area',
            'tight_constraints': True
        }
    ]
    
    results = []
    
    for scenario in scenarios:
        print(f"\n📋 {scenario['name']}: {scenario['description']}")
        print("-" * 50)
        
        # Temporarily modify constraints if needed
        original_is_feasible = is_feasible
        if scenario.get('tight_constraints'):
            def tight_is_feasible(x, y):
                return (1 <= x <= 10 and 1 <= y <= 8 and 
                        not (x >= 7 and y >= 3))  # Tighter restrictions
            globals()['is_feasible'] = tight_is_feasible
        
        # Run heuristic
        center_x, center_y = calculate_center_of_gravity(scenario['locations'])
        candidates = generate_grid_candidates(center_x, center_y)
        
        if candidates:
            best_candidate, grid_cost = evaluate_candidates(scenario['locations'], candidates)
            final_loc, final_cost, _ = hill_climbing_improvement(scenario['locations'], best_candidate)
            
            results.append({
                'scenario': scenario['name'],
                'final_location': final_loc,
                'final_cost': final_cost,
                'candidates_generated': len(candidates),
                'center_feasible': is_feasible(center_x, center_y)
            })
            
            print(f"✅ Solution found: ({final_loc[0]:.3f}, {final_loc[1]:.3f})")
            print(f"✅ Final cost: {final_cost:.2f}")
        else:
            print(f"❌ No feasible candidates found!")
            results.append({
                'scenario': scenario['name'],
                'final_location': None,
                'final_cost': float('inf'),
                'candidates_generated': 0,
                'center_feasible': False
            })
        
        # Restore original constraints
        globals()['is_feasible'] = original_is_feasible
    
    # Summary table
    print("\n" + "="*60)
    print("ROBUSTNESS TESTING SUMMARY")
    print("="*60)
    print(f"{'Scenario':<20} {'Location':<20} {'Cost':<10} {'Candidates':<12} {'Center Feasible':<15}")
    print("-" * 80)
    
    for result in results:
        if result['final_location']:
            loc_str = f"({result['final_location'][0]:.2f}, {result['final_location'][1]:.2f})"
            cost_str = f"{result['final_cost']:.0f}"
        else:
            loc_str = "No solution"
            cost_str = "N/A"
        
        center_feas_str = "Yes" if result['center_feasible'] else "No"
        
        print(f"{result['scenario']:<20} {loc_str:<20} {cost_str:<10} {result['candidates_generated']:<12} {center_feas_str:<15}")

# Test robustness scenarios
test_robustness_scenarios()

### Why This Tier Exists vs Tier 1

**Tier 2 (Geometric Iterative Refinement) addresses Tier 1 limitations:**
- **Real-world constraints** - Handles restricted areas and boundaries
- **Discrete feasibility** - Considers practical, implementable locations
- **Local optimization** - Refines solution within feasible region
- **Robustness** - Works when mathematical optimum is infeasible

**Key innovations over Tier 1:**
- **Feasibility filtering** - Eliminates impractical locations
- **Grid search** - Systematic exploration of solution space
- **Hill climbing** - Local improvement mechanism
- **Constraint handling** - Adaptable to various business rules

### Pros vs Cons

**Pros:**
- ✅ Handles real-world constraints and restrictions
- ✅ Produces practical, implementable solutions
- ✅ Transparent and explainable process
- ✅ Customizable feasibility rules
- ✅ Good performance on most practical problems
- ✅ Robust to problem variations

**Cons:**
- ❌ No optimality guarantee (heuristic)
- ❌ Performance depends on parameter tuning
- ❌ May get stuck in local optima
- ❌ Computationally heavier than Tier 1
- ❌ Solution quality varies with constraint complexity

### When to Use This Tier

**Ideal for:**
- **Constrained environments** - When restrictions apply to facility location
- **Practical implementation** - When solution must be buildable
- **Regulatory compliance** - Zoning, environmental, or legal constraints
- **Discrete locations** - When only specific sites are feasible
- **Business rules** - When operational constraints exist

**Use when:**
- Mathematical optimum is infeasible or impractical
- You need to respect geographical or regulatory constraints
- Solution must be implementable in the real world
- Multiple feasibility criteria must be satisfied
- You want a balance between optimality and practicality

**Consider Tier 3 when:**
- Problem has complex, non-linear constraints
- Multiple objectives need simultaneous optimization
- Solution space is very large or complex
- You need more sophisticated search capabilities

### Key Takeaways

The geometric iterative refinement heuristic bridges the gap between mathematical theory and practical application. By incorporating feasibility constraints and local improvement, it provides solutions that work in the real world while maintaining good solution quality. The method's transparency and adaptability make it valuable for practitioners who need to balance optimality with practical constraints.